## Adding An Agent to the RAG Pipeline

In [1]:
# Importing libraries and create openai client
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import requests

In [2]:
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
# Asking without function tools
messages = [
    {"role": "user", "content": "Can I still join the course?"}
]

response = openai_client.chat.completions.create(
    model="qwen/qwen3.5-9b",
    messages=messages
)

response.choices[0].message.content

'That depends on the specific details of the course and the institution offering it. To give you a helpful answer, could you clarify:\n\n1. **Current date vs. registration deadline**: Are you asking about an upcoming start date, or did you miss the official enrollment period?  \n2. **Course type**: Is this an online/self-paced course, a semester-based class, a workshop, etc.?  \n3. **Institution policies**: Some programs allow late registration with fees, while others have strict deadlines.  \n4. **Your situation**: Are you asking because of work conflicts, relocation, financial reasons, or something else?  \n\nMany courses still accept enrollments after deadlines (e.g., audit options, waitlists, or special circumstances), but this varies widely. Let me know more details, and I can help guide your next steps! 😊'

Note: general response is provided due to lack of context. RAG is needed here and having a search tool called also provides a more accurate response

In [5]:
# Defining the tool - define a search function that queries the index directly

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
# Next step is to tell the LLM model about this function tool
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [7]:
# Now let's ask the same question with the search tool implemented - response is blank because tool call being implemented here as intended
response = openai_client.chat.completions.create(
    model="qwen/qwen3.5-9b",
    messages=messages,
    tools=[search_tool],
)

In [8]:
# Need to parse arguments to identify the tool_call or search function in this case
response

ChatCompletion(id='chatcmpl-65dvic5g224a178a09trxp', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='915049670', function=Function(arguments='{"query":"join course"}', name='search'), type='function')], reasoning_content=''))], created=1781783990, model='qwen/qwen3.5-9b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='qwen/qwen3.5-9b', usage=CompletionUsage(completion_tokens=25, prompt_tokens=295, total_tokens=320, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=None), stats={})

In [9]:
# need to send this info back to the LLM
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='915049670', function=Function(arguments='{"query":"join course"}', name='search'), type='function')

Steps
- make a call to the LLM <--
- LLM decided to invoke search('params')
- We invoke the search, we have the results
- Send the results back to the LLM (another call) <--
- LLM processes the results
- LLM gives the answer


In [10]:
import json

call = response.choices[0].message.tool_calls[0]
args_dict = json.loads(call.function.arguments)

results = search(**args_dict)
result_json = json.dumps(results, indent=2)

In [11]:
print(call)

ChatCompletionMessageFunctionToolCall(id='915049670', function=Function(arguments='{"query":"join course"}', name='search'), type='function')


In [ ]:
print(call.function)

Function(arguments='{"query":"join the course late registration deadline eligibility requirements"}', name='search')


In [12]:
args_dict

{'query': 'join course'}

Now we send this result back to the model. we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.

In [13]:
# Initial message history
messages

[{'role': 'user', 'content': 'Can I still join the course?'}]

In [14]:
# Need to append LLM's response info to messages history
messages.append({
    "role": "assistant",
    "content": response.choices[0].message.content,
})

print(response.choices[0].message.content) # Sometimes the content is blank

In [15]:
# Need to send search results to messages history as well
messages.append({
    "role": "tool",
    "tool_call_id": call.id, # for the response API you use call.call_id
    "content": result_json,
})

print(result_json) # this info will be sent to the LLM

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "When will the course be offered next?",
    "answer": "Summer 2027."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

In [16]:
# Updated message history
messages

[{'role': 'user', 'content': 'Can I still join the course?'},
 {'role': 'assistant', 'content': ''},
 {'role': 'tool',
  'tool_call_id': '915049670',
  'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/),

In [17]:
# Asking the model once more with added context from search tool
response = openai_client.chat.completions.create(
    model="qwen/qwen3.5-9b",
    messages=messages,
    tools=[search_tool],
)

response.choices[0].message.content

'Yes, you can still join the course!\n\nHere are a few important details based on the latest information:\n\n*   **Certificate:** You can receive a certificate, but you must submit your project while submissions are still being accepted. This means you need to join a "live" cohort rather than studying entirely on your own pace.\n*   **Registration:** There is no official waiting list or deadline cutoff mentioned in the FAQ; it suggests you can start learning and submitting homework even before the course officially starts.\n*   **Next Offering:** If you are looking ahead, the next course will be offered in Summer 2027.\n\nTo get started, the recommended workflow is to watch the lesson videos, work through the GitHub notebooks, and read the homework instructions available on their documentation site and GitHub repository.'

In [18]:
print(response)

ChatCompletion(id='chatcmpl-wvliyeycwkxk5z6paef0d', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, you can still join the course!\n\nHere are a few important details based on the latest information:\n\n*   **Certificate:** You can receive a certificate, but you must submit your project while submissions are still being accepted. This means you need to join a "live" cohort rather than studying entirely on your own pace.\n*   **Registration:** There is no official waiting list or deadline cutoff mentioned in the FAQ; it suggests you can start learning and submitting homework even before the course officially starts.\n*   **Next Offering:** If you are looking ahead, the next course will be offered in Summer 2027.\n\nTo get started, the recommended workflow is to watch the lesson videos, work through the GitHub notebooks, and read the homework instructions available on their documentation site and GitHub repository.', refusal=None,

Note: when using the chat completions API you cannot send python objects in the content field. You would either need to send the message content (if any) from the assistant or reformat the content field to send as a string. It is also required to have role added for each instance in the message history.

### Calculating Usage Cost - Using OpenAI models

In [74]:
response

ChatCompletion(id='chatcmpl-1a0ys6xtf7igvxwu88dus', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, you can still join the course! The videos, GitHub materials, and registration forms are available, so you can start learning whenever you want.\n\nHowever, please note that if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted (which happens during the "live" cohort period). Certificates are only awarded for completing the course with a live group, as peer-reviewing projects can only be done while the course is running.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=''))], created=1781699484, model='qwen/qwen3.5-9b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='qwen/qwen3.5-9b', usage=CompletionUsage(completion_tokens=98, prompt_tokens=1032, total_tokens=1130, co

In [75]:
usage = response.usage
usage.prompt_tokens, usage.completion_tokens

(1032, 98)

In [76]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.prompt_tokens, usage.completion_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0002136
